# EL Image Classification — All-in-One Notebook

This notebook contains the full pipeline in one file:
1. Data preparation and DataLoaders
2. Transfer-learning training (ResNet18)
3. ONNX export
4. Edge-style inference and optional MQTT publish

## Usage Notes
- Run cells top-to-bottom.
- Keep `PVEL-AD.zip` in the project root (or update `ZIP_PATH`).
- Set `INFERENCE_IMAGE_PATH` before running inference.
- MQTT publishing is optional and disabled by default.

In [1]:
from __future__ import annotations

import copy
import json
import random
import zipfile
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import cv2
import numpy as np
import onnx
import onnxruntime as ort
import paho.mqtt.client as mqtt
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm import tqdm

In [2]:
# ===== Configuration =====
ZIP_PATH = Path('PVEL-AD.zip')
EXTRACT_DIR = Path('data')
CHECKPOINT_PATH = Path('best_model.pth')
ONNX_OUTPUT_PATH = Path('best_model.onnx')

EPOCHS = 20
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5
NUM_WORKERS = 2
IMAGE_SIZE = 224
SEED = 42

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CLASS_NAMES = ['Defective', 'Functional']
CLASS_TO_INDEX = {'Defective': 0, 'Functional': 1}
VALID_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

print(f'Device: {DEVICE}')

Device: cpu


In [3]:
# ===== Data pipeline =====
@dataclass
class SplitItems:
    train: List[Tuple[Path, int]]
    val: List[Tuple[Path, int]]
    test: List[Tuple[Path, int]]


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def extract_dataset(zip_path: Path, extract_dir: Path) -> Path:
    extract_dir.mkdir(parents=True, exist_ok=True)

    existing_images = list(extract_dir.rglob('*'))
    if any(p.is_file() and p.suffix.lower() in VALID_EXTENSIONS for p in existing_images):
        return extract_dir

    if not zip_path.exists():
        raise FileNotFoundError(f'ZIP dataset not found: {zip_path}')

    with zipfile.ZipFile(zip_path, 'r') as archive:
        archive.extractall(extract_dir)

    return extract_dir


def _infer_label_from_path(path: Path) -> int:
    parts_lower = {part.lower() for part in path.parts}
    if 'defective' in parts_lower:
        return CLASS_TO_INDEX['Defective']
    if 'functional' in parts_lower:
        return CLASS_TO_INDEX['Functional']
    raise ValueError(f'Unable to infer class label from path: {path}')


def discover_images(root_dir: Path) -> List[Tuple[Path, int]]:
    samples: List[Tuple[Path, int]] = []

    for file_path in root_dir.rglob('*'):
        if not file_path.is_file() or file_path.suffix.lower() not in VALID_EXTENSIONS:
            continue
        try:
            label = _infer_label_from_path(file_path)
        except ValueError:
            continue
        samples.append((file_path, label))

    if not samples:
        raise RuntimeError('No valid classed images found. Ensure ZIP contains Defective/Functional folders.')

    labels = [label for _, label in samples]
    for class_name, class_index in CLASS_TO_INDEX.items():
        if class_index not in labels:
            raise RuntimeError(f"Missing class '{class_name}' in extracted dataset under {root_dir}")

    return sorted(samples, key=lambda item: str(item[0]))


def stratified_split(
    samples: Sequence[Tuple[Path, int]],
    train_ratio: float = 0.70,
    val_ratio: float = 0.15,
    test_ratio: float = 0.15,
    seed: int = 42,
) -> SplitItems:
    if not np.isclose(train_ratio + val_ratio + test_ratio, 1.0):
        raise ValueError('Split ratios must sum to 1.0')

    rng = random.Random(seed)
    by_class: Dict[int, List[Tuple[Path, int]]] = {0: [], 1: []}
    for item in samples:
        by_class[item[1]].append(item)

    train_items: List[Tuple[Path, int]] = []
    val_items: List[Tuple[Path, int]] = []
    test_items: List[Tuple[Path, int]] = []

    for class_index, class_items in by_class.items():
        if len(class_items) < 3:
            raise RuntimeError(
                f'Class index {class_index} has too few samples ({len(class_items)}) for 70/15/15 split.'
            )

        shuffled = class_items[:]
        rng.shuffle(shuffled)

        n_total = len(shuffled)
        n_train = max(1, int(n_total * train_ratio))
        n_val = max(1, int(n_total * val_ratio))
        n_test = n_total - n_train - n_val

        if n_test < 1:
            n_test = 1
            if n_train > n_val:
                n_train -= 1
            else:
                n_val -= 1

        train_items.extend(shuffled[:n_train])
        val_items.extend(shuffled[n_train:n_train + n_val])
        test_items.extend(shuffled[n_train + n_val:])

    rng.shuffle(train_items)
    rng.shuffle(val_items)
    rng.shuffle(test_items)

    return SplitItems(train=train_items, val=val_items, test=test_items)


def build_transforms(image_size: int = 224):
    train_tfms = transforms.Compose([
        transforms.Grayscale(num_output_channels=3),
        transforms.Resize((image_size, image_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

    eval_tfms = transforms.Compose([
        transforms.Grayscale(num_output_channels=3),
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

    return train_tfms, eval_tfms, eval_tfms


class ELDataset(Dataset):
    def __init__(self, items: Sequence[Tuple[Path, int]], transform: transforms.Compose):
        self.items = list(items)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, idx: int):
        image_path, label = self.items[idx]

        image_gray = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
        if image_gray is None:
            raise RuntimeError(f'Failed to read image: {image_path}')

        image_denoised = cv2.medianBlur(image_gray, 3)
        image_pil = Image.fromarray(image_denoised)
        image_tensor = self.transform(image_pil)

        return image_tensor, label


def create_dataloaders(
    zip_path: Path,
    extract_dir: Path,
    batch_size: int = 32,
    num_workers: int = 2,
    image_size: int = 224,
    seed: int = 42,
):
    set_seed(seed)

    extracted_root = extract_dataset(zip_path, extract_dir)
    all_samples = discover_images(extracted_root)
    split_items = stratified_split(all_samples, 0.70, 0.15, 0.15, seed)

    train_tfms, val_tfms, test_tfms = build_transforms(image_size=image_size)

    train_ds = ELDataset(split_items.train, train_tfms)
    val_ds = ELDataset(split_items.val, val_tfms)
    test_ds = ELDataset(split_items.test, test_tfms)

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
    )

    split_counts = {
        'train': len(train_ds),
        'val': len(val_ds),
        'test': len(test_ds),
        'total': len(all_samples),
    }

    return train_loader, val_loader, test_loader, CLASS_TO_INDEX, split_counts

In [4]:
# ===== Model, evaluation, and training =====
def build_model(num_classes: int = 2, freeze_early_layers: bool = True) -> nn.Module:
    weights = models.ResNet18_Weights.DEFAULT
    model = models.resnet18(weights=weights)

    if freeze_early_layers:
        modules_to_freeze = [
            model.conv1,
            model.bn1,
            model.layer1,
            model.layer2,
        ]
        for module in modules_to_freeze:
            for param in module.parameters():
                param.requires_grad = False

    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)

    return model


def evaluate(model: nn.Module, data_loader, criterion, device: torch.device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    tp = fp = tn = fn = 0

    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            preds = torch.argmax(outputs, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

            tp += ((preds == 0) & (labels == 0)).sum().item()
            fp += ((preds == 0) & (labels == 1)).sum().item()
            tn += ((preds == 1) & (labels == 1)).sum().item()
            fn += ((preds == 1) & (labels == 0)).sum().item()

    avg_loss = running_loss / max(total, 1)
    accuracy = correct / max(total, 1)
    confusion = {'tp': tp, 'fp': fp, 'tn': tn, 'fn': fn}
    return avg_loss, accuracy, confusion


def train_model(
    zip_path: Path,
    extract_dir: Path,
    checkpoint_path: Path,
    epochs: int = 20,
    batch_size: int = 32,
    learning_rate: float = 1e-4,
    weight_decay: float = 1e-5,
    num_workers: int = 2,
    image_size: int = 224,
    seed: int = 42,
    device: Optional[torch.device] = None,
):
    set_seed(seed)
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    train_loader, val_loader, test_loader, class_to_idx, split_counts = create_dataloaders(
        zip_path=zip_path,
        extract_dir=extract_dir,
        batch_size=batch_size,
        num_workers=num_workers,
        image_size=image_size,
        seed=seed,
    )

    model = build_model(num_classes=2, freeze_early_layers=True).to(device)
    criterion = nn.CrossEntropyLoss()
    trainable_params = [param for param in model.parameters() if param.requires_grad]
    optimizer = optim.Adam(trainable_params, lr=learning_rate, weight_decay=weight_decay)

    best_val_acc = -1.0
    best_state = None

    print(f'Device: {device}')
    print(f'Split counts: {split_counts}')
    print(f'Class mapping: {class_to_idx}')

    for epoch in range(epochs):
        model.train()
        running_train_loss = 0.0
        n_train = 0

        train_progress = tqdm(train_loader, desc=f'Epoch {epoch + 1}/{epochs} [Train]', leave=False)
        for images, labels in train_progress:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad(set_to_none=True)
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            current_batch_size = images.size(0)
            running_train_loss += loss.item() * current_batch_size
            n_train += current_batch_size

            train_progress.set_postfix(loss=f'{loss.item():.4f}')

        train_loss = running_train_loss / max(n_train, 1)
        val_loss, val_acc, _ = evaluate(model, val_loader, criterion, device)

        print(
            f'Epoch {epoch + 1}/{epochs} | '
            f'Train Loss: {train_loss:.4f} | '
            f'Val Loss: {val_loss:.4f} | '
            f'Val Acc: {val_acc:.4f}'
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())
            checkpoint = {
                'model_state_dict': best_state,
                'class_to_idx': class_to_idx,
                'image_size': image_size,
                'best_val_acc': best_val_acc,
            }
            torch.save(checkpoint, checkpoint_path)
            print(f'Saved improved checkpoint to: {checkpoint_path}')

    if best_state is None:
        raise RuntimeError('Training completed but no best model state was captured.')

    model.load_state_dict(best_state)
    test_loss, test_acc, confusion = evaluate(model, test_loader, criterion, device)
    print(
        f'Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f} | '
        f'Confusion (Defective=positive): {confusion}'
    )

    return model, class_to_idx, split_counts, best_val_acc

In [5]:
# ===== ONNX export =====
def export_to_onnx(
    checkpoint_path: Path,
    onnx_output: Path,
    image_size: int = 224,
    opset_version: int = 17,
) -> None:
    if not checkpoint_path.exists():
        raise FileNotFoundError(f'Checkpoint not found: {checkpoint_path}')

    checkpoint = torch.load(checkpoint_path, map_location='cpu')

    model = build_model(num_classes=2, freeze_early_layers=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()

    dummy_input = torch.randn(1, 3, image_size, image_size, dtype=torch.float32)

    onnx_output.parent.mkdir(parents=True, exist_ok=True)

    torch.onnx.export(
        model,
        dummy_input,
        str(onnx_output),
        export_params=True,
        opset_version=opset_version,
        do_constant_folding=True,
        input_names=['input'],
        output_names=['logits'],
        dynamic_axes={
            'input': {0: 'batch_size'},
            'logits': {0: 'batch_size'},
        },
    )

    loaded = onnx.load(str(onnx_output))
    onnx.checker.check_model(loaded)

    print(f'ONNX export complete: {onnx_output}')

In [6]:
# ===== Inference + optional MQTT =====
IMAGENET_MEAN_NP = np.array(IMAGENET_MEAN, dtype=np.float32)
IMAGENET_STD_NP = np.array(IMAGENET_STD, dtype=np.float32)


def softmax(logits: np.ndarray) -> np.ndarray:
    logits = logits - np.max(logits, axis=1, keepdims=True)
    exp_values = np.exp(logits)
    return exp_values / np.sum(exp_values, axis=1, keepdims=True)


def preprocess_el_image(image_path: Path, image_size: int = 224) -> np.ndarray:
    gray = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if gray is None:
        raise FileNotFoundError(f'Unable to read image: {image_path}')

    denoised = cv2.medianBlur(gray, 3)
    rgb = np.stack([denoised, denoised, denoised], axis=-1)
    resized = cv2.resize(rgb, (image_size, image_size), interpolation=cv2.INTER_AREA)

    normalized = resized.astype(np.float32) / 255.0
    normalized = (normalized - IMAGENET_MEAN_NP) / IMAGENET_STD_NP

    chw = np.transpose(normalized, (2, 0, 1))
    batched = np.expand_dims(chw, axis=0).astype(np.float32)
    return batched


def infer_severity_score(onnx_model_path: Path, image_path: Path, image_size: int = 224) -> float:
    providers = ['CPUExecutionProvider']
    session = ort.InferenceSession(str(onnx_model_path), providers=providers)

    input_name = session.get_inputs()[0].name
    model_input = preprocess_el_image(image_path=image_path, image_size=image_size)

    outputs = session.run(None, {input_name: model_input})
    logits = outputs[0]
    probabilities = softmax(logits)
    defect_probability = float(probabilities[0, CLASS_TO_INDEX['Defective']])
    return defect_probability


def build_payload(pad_id: str, severity_score: float, critical_threshold: float = 0.80) -> Dict[str, object]:
    status = 'CRITICAL' if severity_score > critical_threshold else 'OK'
    return {
        'pad_id': pad_id,
        'severity_score': round(severity_score, 4),
        'status': status,
    }


def publish_mqtt(payload: Dict[str, object], broker_host: str, broker_port: int, topic: str) -> None:
    client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
    client.connect(broker_host, broker_port, keepalive=30)
    client.publish(topic, json.dumps(payload), qos=0, retain=False)
    client.disconnect()

## Run Training
Set `RUN_TRAINING = True` to start training. This can take significant time.

In [7]:
RUN_TRAINING = True

if RUN_TRAINING:
    model, class_to_idx, split_counts, best_val_acc = train_model(
        zip_path=ZIP_PATH,
        extract_dir=EXTRACT_DIR,
        checkpoint_path=CHECKPOINT_PATH,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        num_workers=NUM_WORKERS,
        image_size=IMAGE_SIZE,
        seed=SEED,
        device=DEVICE,
    )
    print(f'Best validation accuracy: {best_val_acc:.4f}')
else:
    print('Training skipped. Set RUN_TRAINING = True to train.')

RuntimeError: No valid classed images found. Ensure ZIP contains Defective/Functional folders.

## Export ONNX
Set `RUN_EXPORT = True` after you have a checkpoint file.

In [ ]:
RUN_EXPORT = False

if RUN_EXPORT:
    export_to_onnx(
        checkpoint_path=CHECKPOINT_PATH,
        onnx_output=ONNX_OUTPUT_PATH,
        image_size=IMAGE_SIZE,
        opset_version=17,
    )
else:
    print('ONNX export skipped. Set RUN_EXPORT = True to export.')

## Run Inference
Set `INFERENCE_IMAGE_PATH` to a real EL image path, then set `RUN_INFERENCE = True`.

In [ ]:
RUN_INFERENCE = False
INFERENCE_IMAGE_PATH = Path('path/to/el_image.png')
PAD_ID = 'simulated_pad_01'
CRITICAL_THRESHOLD = 0.80

MQTT_ENABLE = False
MQTT_BROKER = 'localhost'
MQTT_PORT = 1883
MQTT_TOPIC = 'pv/inspection/severity'

if RUN_INFERENCE:
    if not ONNX_OUTPUT_PATH.exists():
        raise FileNotFoundError(f'ONNX model not found: {ONNX_OUTPUT_PATH}')

    score = infer_severity_score(
        onnx_model_path=ONNX_OUTPUT_PATH,
        image_path=INFERENCE_IMAGE_PATH,
        image_size=IMAGE_SIZE,
    )

    payload = build_payload(
        pad_id=PAD_ID,
        severity_score=score,
        critical_threshold=CRITICAL_THRESHOLD,
    )

    print(json.dumps(payload, indent=2))

    if MQTT_ENABLE:
        try:
            publish_mqtt(
                payload=payload,
                broker_host=MQTT_BROKER,
                broker_port=MQTT_PORT,
                topic=MQTT_TOPIC,
            )
            print('MQTT publish: success')
        except Exception as exc:
            print(f'MQTT publish failed (non-blocking): {exc}')
else:
    print('Inference skipped. Set RUN_INFERENCE = True and INFERENCE_IMAGE_PATH to run.')